## Portfolio 1 - Extracting linguistic features using spaCy

*By Sofie Mosegaard*

This assignment concerns using spaCy to extract linguistic information from a corpus of texts.

The corpus is an interesting one: The Uppsala Student English Corpus (USE). All of the data is included in the folder called in but you can access more documentation via this link: https://ota.bodleian.ox.ac.uk/repository/xmlui/handle/20.500.12024/2457.

*Objective* - this assignment is designed to test that you can:

1. Work with multiple input data arranged hierarchically in folders;
2. Use spaCy to extract linguistic information from text data;
3. Save those results in a clear way which can be shared or used for future analysis

In [2]:
# Import libraries

import os
import pandas as pd
import glob
import re

import spacy
# python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")


In [3]:
# First, specify the filepath to the folder with all the data
filepath = os.path.join(
                        "..",
                        "in",
                        "USEcorpus"
                        )

# Loop over each of the 14 subfolders (a1, a2, a3...)
for subfolder in sorted(os.listdir(filepath)): # sorted = loops through the subfolders in the original, sorted order
    subfolder_path = os.path.join(filepath, subfolder)

    if os.path.isdir(subfolder_path): # Check if the specified directory exists or nor

        # Initialize empty lists to store data  
        filenames = [] # Original name of the text files in the subfolders
        nouns_freq = []
        verbs_freq = []
        adverbs_freq = []
        adjectives_freq = []
        no_unique_per = []
        no_unique_loc = []
        no_unique_org = []

    # Loop over each text file in the subfolder
        for file in glob.glob(os.path.join(subfolder_path, "*.txt")):
            if os.path.isfile(file): # Function checks whether there excists files on the specified path
                with open(file, "r", encoding = "latin-1") as f:
                    text = f.read()

                    text = re.sub(r"<*?>", "", text) # Remove metadata between <> and replace it with "" (= nothing hehe)

                    doc = nlp(text) # Create spacy doc

                    ### Count number of each POS ###

                    # Count number of nouns
                    nouns_count = 0
                    for token in doc:
                        if token.pos_ == "NOUN":
                            nouns_count += 1

                    # count number of verbs
                    verbs_count = 0
                    for token in doc:
                        if token.pos_ == "VERB":
                            verbs_count += 1

                    # count number of adverbs
                    adverb_count = 0
                    for token in doc:
                        if token.pos_ == "ADV":
                            adverb_count += 1

                    # count number of adjectives
                    adjective_count = 0
                    for token in doc:
                        if token.pos_ == "ADJ":
                            adjective_count += 1
                
                    # Calculate their relative frequency per 10,000 words
                    nouns_relative_freq = (nouns_count/len(doc)) * 10000
                    verbs_relative_freq = (verbs_count/len(doc)) * 10000
                    adverb_relative_freq = (adverb_count/len(doc)) * 10000
                    adjective_relative_freq = (adjective_count/len(doc)) * 10000
                    
                    # Round the decimals
                    nouns_relative_freq = round(nouns_relative_freq, 2)
                    verbs_relative_freq = round(verbs_relative_freq, 2)
                    adverb_relative_freq = round(adverb_relative_freq, 2)
                    adjective_relative_freq = round(adjective_relative_freq, 2)

                    ### Count total number of unique PER, LOC, and ORG entities ###

                    unique_per_count = 0
                    unique_loc_count = 0
                    unique_org_count = 0

                    # Iterate over each entity in the spacy doc
                    for ent in doc.ents:
                        # Check the entity label --> if it is unique, then increment the corresponding count
                        if ent.label_ == "PERSON":
                            unique_per_count += 1
                        elif ent.label_ == "LOC":
                            unique_loc_count += 1
                        elif ent.label_ == "ORG":
                            unique_org_count += 1
                    
                    # Append the relative frequency of POS and counts of unique entities to the lists
                    filenames.append(os.path.basename(file))
                    nouns_freq.append(nouns_relative_freq)
                    verbs_freq.append(verbs_relative_freq)
                    adverbs_freq.append(adverb_relative_freq)
                    adjectives_freq.append(adjective_relative_freq)
                    no_unique_per.append(unique_per_count)
                    no_unique_loc.append(unique_loc_count)
                    no_unique_org.append(unique_org_count)
    

    # Create a pandas dataframe for each subfolder
    df = pd.DataFrame({
        "Filename": filenames, 
        "Nouns_Relative_Freq": nouns_freq,
        "Verbs_Relative_Freq": verbs_freq,
        "Adverbs_Relative_Freq": adverbs_freq,
        "No_unique_per": no_unique_per,
        "No_unique_loc": no_unique_loc,
        "No_unique_org": no_unique_org
    })

    # Save the dataframe as a .csv file
    csv_filename = f"../out/{subfolder}_data.csv"
    df.to_csv(csv_filename)

In [4]:
# Import one .csv file to check the table

data = pd.read_csv("../out/a4_data.csv")
data

,Unnamed: 0,Filename,Nouns_Relative_Freq,Verbs_Relative_Freq,Adverbs_Relative_Freq,No_unique_per,No_unique_loc,No_unique_org
0,0,3062.a4.txt,1729.13,954.00,485.52,16,0,4
1,1,2030.a4.txt,1395.55,1258.56,419.52,37,0,2
2,2,0158.a4.txt,1399.63,1086.56,561.69,23,0,1
3,3,2066.a4.txt,1678.74,1099.03,446.86,53,0,3
4,4,0238.a4.txt,1487.76,1299.44,310.73,30,0,1
...,...,...,...,...,...,...,...,...
180,180,0194.a4.txt,1675.03,1173.52,391.17,9,0,11
181,181,1104.a4.txt,1625.62,975.37,334.98,51,0,4
182,182,0179.a4.txt,1756.62,1040.24,215.90,34,0,2
183,183,2049.a4.txt,1878.57,892.86,300.00,15,0,35
